## In this lab, we build a banking data model in Databricks using Unity Catalog and Delta Lake. We create dimension and fact tables, implement SCD Type 2 to track customer history, use Change Data Feed (CDF) to monitor data changes, and leverage Delta Time Travel to query and restore previous table versions. The lab demonstrates data governance, auditing, historical tracking, and compliance reporting capabilities.

%md
## **Before starting the lab, you need a Unity Catalog volume to store the file**. Follow these steps to create one:

- In the Databricks workspace sidebar, click Catalog.
- In the Catalog Explorer,right side click on + icon and create a Catalog ( DP750_Lab_06 ), type =standard & default storage.
- Then Create expand Catalog and generate a new schema (Silver) by clicking on + icon and create Schema.
- Click the ⋮ menu next to default, then select Create > Volume. (Bronze)
- Enter lab_data as the volume name, leave the type as Managed, and click Create.

## Exercise 1: Create the Northbank Financial Data Model

In this exercise, you design the foundational Delta tables for Northbank Financial's analytics platform. The model includes a **customer dimension** designed for SCD Type 2 history tracking, and a **transaction fact table** optimised with liquid clustering and Change Data Feed.

After completing this exercise, return to the lab setup page to explore the tables you created in **Catalog Explorer**.

### Task 1.2 — Create the `dim_customer` SCD Type 2 table

Create a managed Delta table `banking_lab.silver.dim_customer`. This table tracks **customer dimension history** using the SCD Type 2 pattern.

The table must include:

| Column | Type | Notes |
|---|---|---|
| `customer_sk` | `BIGINT GENERATED ALWAYS AS IDENTITY` | Surrogate key — auto-incrementing, unique per version |
| `customer_id` | `STRING NOT NULL` | Business/natural key from source system |
| `full_name` | `STRING` | |
| `email` | `STRING` | |
| `city` | `STRING` | City of primary residence |
| `segment` | `STRING` | Customer segment: Retail, Premium, Business |
| `account_type` | `STRING` | Account type: Current, Savings, Business |
| `valid_from` | `TIMESTAMP NOT NULL` | When this version became effective |
| `valid_to` | `TIMESTAMP NOT NULL` | When this version expired (9999-12-31 for active records) |
| `is_current` | `BOOLEAN` | True for the active version |

Apply **liquid clustering** on `customer_id` and enable **Change Data Feed**.


spark.sql("""
    CREATE TABLE IF NOT EXISTS lab6.silver.dim_customer (
        customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
        customer_id STRING NOT NULL,
        full_name STRING,
        email STRING,
        city STRING,
        segment STRING,
        account_type STRING,
        valid_from TIMESTAMP NOT NULL,
        valid_to TIMESTAMP NOT NULL,
        is_current BOOLEAN
    )
    USING DELTA
    CLUSTER BY (customer_id)
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

Explanation:

- `CREATE TABLE IF NOT EXISTS` creates the table only if it does not already exist.
- `lab6.silver.dim_customer` is the table name.
- `dim_customer` is a dimension table that stores customer information.

--------------------------------------------------

Columns:

- `customer_sk BIGINT GENERATED ALWAYS AS IDENTITY`
  - Surrogate key generated automatically.
  - Unique identifier for each customer record.

- `customer_id STRING NOT NULL`
  - Business/customer identifier.
  - Cannot be NULL.

- `full_name`, `email`, `city`, `segment`, `account_type`
  - Customer attributes stored as strings.

- `valid_from TIMESTAMP NOT NULL`
  - Record start date/time.

- `valid_to TIMESTAMP NOT NULL`
  - Record end date/time.

- `is_current BOOLEAN`
  - Indicates whether this is the latest active record.

--------------------------------------------------

USING DELTA

Explanation:

- Creates the table in Delta Lake format.
- Provides ACID transactions, versioning, and better reliability.

Purpose:
Store data as a Delta table with advanced lakehouse features.

--------------------------------------------------

CLUSTER BY (customer_id)

Explanation:

- Organizes data based on `customer_id`.
- Improves query performance when filtering or searching by customer_id.

Purpose:
Speed up customer-related queries.

--------------------------------------------------

TBLPROPERTIES (delta.enableChangeDataFeed = true)

Explanation:

- Enables Change Data Feed (CDF).
- Tracks inserts, updates, and deletes made to the table.

Purpose:
Allow downstream processes to read only changed records instead of the entire table.

--------------------------------------------------

Purpose:

Create a Delta dimension table to store customer master data with support for historical tracking and efficient querying.

--------------------------------------------------

Overall Purpose:

This command creates the `dim_customer` Delta table in the Silver layer, optimized for customer analytics, historical record management (SCD), and change data tracking.

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS dp750_lab_06.silver.dim_customer (
        customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
        customer_id STRING NOT NULL,
        full_name STRING,
        email STRING,
        city STRING,
        segment STRING,
        account_type STRING,
        valid_from TIMESTAMP NOT NULL,
        valid_to TIMESTAMP NOT NULL,
        is_current BOOLEAN
    )
    USING DELTA
    CLUSTER BY (customer_id)
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

DataFrame[]

### Task 1.3 — Create the `fact_transactions` table

Create a managed Delta table `banking_lab.silver.fact_transactions` to store individual banking payments.

| Column | Type | Notes |
|---|---|---|
| `transaction_id` | `STRING NOT NULL` | Unique transaction reference |
| `account_id` | `STRING` | Bank account identifier |
| `customer_id` | `STRING` | Links to `dim_customer` |
| `transaction_type` | `STRING` | Credit, Debit, or Transfer |
| `amount` | `DECIMAL(15,2)` | Transaction amount |
| `currency` | `STRING DEFAULT 'GBP'` | Currency code |
| `transaction_date` | `DATE` | Date of the transaction |
| `description` | `STRING` | Transaction narrative |

Apply **liquid clustering** on `customer_id, transaction_date` to support the expected query patterns (filtering by customer and/or date range). Enable **Change Data Feed** for FCA compliance auditing.


In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS dp750_lab_06.silver.fact_transactions (
        transaction_id STRING NOT NULL,
        account_id STRING,
        customer_id STRING,
        transaction_type STRING,
        amount DECIMAL(15,2),
        currency STRING DEFAULT 'GBP',
        transaction_date DATE,
        description STRING
    )
    USING DELTA
    CLUSTER BY (customer_id, transaction_date)
    TBLPROPERTIES (
        delta.enableChangeDataFeed = true,
        'delta.feature.allowColumnDefaults' = 'supported'
    )
""")

DataFrame[]

## Exercise 2: Implement SCD Type 2 for the Customer Dimension

Northbank's regulatory reporting must reflect the **customer profile at the time of each transaction**, not just the current state. In this exercise, you load initial customer records, apply realistic attribute changes, and implement the **two-step MERGE + INSERT pattern** for SCD Type 2. You then query full customer history and perform a point-in-time lookup.

In [0]:
from pyspark.sql.functions import lit, to_timestamp

initial_customers = [
    ("C-1001", "Emma Hartley",      "emma.hartley@northbank.com",     "London",     "Retail",   "Savings"),
    ("C-1002", "James Weston",      "james.weston@northbank.com",     "Manchester", "Retail",   "Current"),
    ("C-1003", "Sophia Chen",       "sophia.chen@northbank.com",      "Edinburgh",  "Premium",  "Savings"),
    ("C-1004", "Oliver Banks",      "oliver.banks@northbank.com",     "Birmingham", "Retail",   "Current"),
    ("C-1005", "Aisha Patel",       "aisha.patel@northbank.com",      "London",     "Business", "Business"),
    ("C-1006", "Liam Murray",       "liam.murray@northbank.com",      "Leeds",      "Retail",   "Savings"),
    ("C-1007", "Charlotte Wright",  "charlotte.wright@northbank.com", "Bristol",    "Premium",  "Current"),
    ("C-1008", "Noah Thompson",     "noah.thompson@northbank.com",    "Glasgow",    "Retail",   "Savings"),
]

df_initial = spark.createDataFrame(
    initial_customers,
    ["customer_id", "full_name", "email", "city", "segment", "account_type"]
)

df_initial = (
    df_initial
    .withColumn("valid_from",  to_timestamp(lit("2020-01-01 00:00:00")))
    .withColumn("valid_to",    to_timestamp(lit("9999-12-31 00:00:00")))
    .withColumn("is_current",  lit(True))
)

# TODO: Write df_initial to banking_lab.silver.dim_customer (append mode)

### Task 2.1 — Load the initial customer records

The cell below defines the **initial dataset** for 8 Northbank customers as a PySpark DataFrame. Your task is to **write this DataFrame to `banking_lab.silver.dim_customer`** in append mode.





from pyspark.sql.functions import lit, to_timestamp

Explanation:

- `lit()` creates a constant value column.
- `to_timestamp()` converts a string into a timestamp data type.
- These functions will be used to create the SCD tracking columns.

--------------------------------------------------

initial_customers = [...]

Explanation:

- A Python list containing sample customer records.
- Each tuple represents one customer.
- Contains:
  - customer_id
  - full_name
  - email
  - city
  - segment
  - account_type

Purpose:
Create sample source data for the customer dimension table.

--------------------------------------------------

df_initial = spark.createDataFrame(...)

Explanation:

- Converts the Python list into a Spark DataFrame.
- The second argument defines the column names.

Purpose:
Create a structured DataFrame that Spark can process.

--------------------------------------------------

.withColumn("valid_from", to_timestamp(lit("2020-01-01 00:00:00")))

Explanation:

- Adds a `valid_from` column.
- Sets the start date for all customer records.
- Converts the string into a timestamp.

Purpose:
Track when a record becomes active.

--------------------------------------------------

.withColumn("valid_to", to_timestamp(lit("9999-12-31 00:00:00")))

Explanation:

- Adds a `valid_to` column.
- Uses a far-future date to indicate the record is currently active.

Purpose:
Support Slowly Changing Dimension (SCD Type 2) tracking.

--------------------------------------------------

.withColumn("is_current", lit(True))

Explanation:

- Adds a boolean column with value `True`.
- Marks these records as the current active versions.

Purpose:
Identify the latest customer records.

--------------------------------------------------

# TODO: Write df_initial to banking_lab.silver.dim_customer (append mode)

Expected Code:

df_initial.write \
    .mode("append") \
    .saveAsTable("lab6.silver.dim_customer")

Explanation:

- `write` starts the DataFrame write operation.
- `mode("append")` adds records without replacing existing data.
- `saveAsTable()` writes the DataFrame into the Delta table.

Purpose:
Load the initial customer records into the `dim_customer` table.

--------------------------------------------------

Overall Purpose:

This code creates a Spark DataFrame containing initial customer data, adds SCD Type 2 tracking columns (`valid_from`, `valid_to`, `is_current`), and prepares the data to be loaded into the Silver-layer customer dimension table.

In [0]:
df_initial.write.mode("append").saveAsTable("dp750_lab_06.silver.dim_customer")

In [0]:
%sql
-- Verify: you should see 8 customer records
SELECT customer_sk, customer_id, full_name, city, segment, is_current
FROM dp750_lab_06.silver.dim_customer
ORDER BY customer_sk;

customer_sk,customer_id,full_name,city,segment,is_current
1,C-1001,Emma Hartley,London,Retail,true
2,C-1002,James Weston,Manchester,Retail,true
3,C-1003,Sophia Chen,Edinburgh,Premium,true
4,C-1004,Oliver Banks,Birmingham,Retail,true
5,C-1005,Aisha Patel,London,Business,true
6,C-1006,Liam Murray,Leeds,Retail,true
7,C-1007,Charlotte Wright,Bristol,Premium,true
8,C-1008,Noah Thompson,Glasgow,Retail,true


### Task 2.2 — Load staging data with customer changes

The operations team has received the following change notifications:

| Customer | Change |
|---|---|
| **C-1001 Emma Hartley** | Moved from **London → Oxford**, upgraded to **Premium** segment |
| **C-1003 Sophia Chen** | Relocated from **Edinburgh → London** |
| **C-1006 Liam Murray** | Updated email address only (city and segment unchanged) |
| **C-1009 Priya Singh** | **New customer** joining from Cardiff, Retail segment |

Run the cell below to create a staging DataFrame and register it as a temp view. You will use it in the next task.

staging_data = [
    ("C-1001", "Emma Hartley", "emma.hartley@northbank.com", "Oxford", "Premium", "Savings"),
    ...
]

Explanation:

- `staging_data` is a Python list containing new and updated customer records.
- Some customers already exist (C-1001, C-1003, C-1006).
- One customer is new (C-1009).

Purpose:
Simulate incoming customer data that will be compared against the existing dimension table.

--------------------------------------------------

df_staging = spark.createDataFrame(...)

Explanation:

- Converts the Python list into a Spark DataFrame.
- Column names are explicitly defined:
  - customer_id
  - full_name
  - email
  - city
  - segment
  - account_type

Purpose:
Create a structured dataset that Spark can process.

--------------------------------------------------

Changes in the staging data:

- `C-1001`
  - City changed from London → Oxford
  - Segment changed from Retail → Premium

- `C-1003`
  - City changed from Edinburgh → London

- `C-1006`
  - Email changed

- `C-1009`
  - New customer record

Purpose:
Provide sample updates and inserts for SCD processing.

--------------------------------------------------

df_staging.createOrReplaceTempView("staging_customers")

Explanation:

- Creates a temporary SQL view named `staging_customers`.
- Allows the DataFrame to be queried using SQL.

Example:

SELECT * FROM staging_customers

Purpose:
Make the staging data available for SQL operations such as MERGE statements.

--------------------------------------------------

print("Staging data loaded. Temp view 'staging_customers' is available.")

Explanation:

- Displays a confirmation message.
- Helps verify that the staging data was loaded successfully.

Purpose:
Provide user feedback.

--------------------------------------------------

display(df_staging)

Explanation:

- Displays the DataFrame in a Databricks table format.
- Lets you visually inspect the incoming records.

Purpose:
Verify the staging data before applying updates.

--------------------------------------------------

Overall Purpose:

This code creates a staging dataset containing customer updates and new customers, converts it into a Spark DataFrame, registers it as a temporary SQL view, and displays it for validation before performing SCD or MERGE operations.

In [0]:
staging_data = [
    ("C-1001", "Emma Hartley",    "emma.hartley@northbank.com",     "Oxford",   "Premium",  "Savings"),
    ("C-1003", "Sophia Chen",     "sophia.chen@northbank.com",      "London",   "Premium",  "Savings"),
    ("C-1006", "Liam Murray",     "liam.new@northbank.com",         "Leeds",    "Retail",   "Savings"),
    ("C-1009", "Priya Singh",     "priya.singh@northbank.com",      "Cardiff",  "Retail",   "Current"),
]

df_staging = spark.createDataFrame(
    staging_data,
    ["customer_id", "full_name", "email", "city", "segment", "account_type"]
)
df_staging.createOrReplaceTempView("staging_customers")
print("Staging data loaded. Temp view 'staging_customers' is available.")
display(df_staging)

Staging data loaded. Temp view 'staging_customers' is available.


customer_id,full_name,email,city,segment,account_type
C-1001,Emma Hartley,emma.hartley@northbank.com,Oxford,Premium,Savings
C-1003,Sophia Chen,sophia.chen@northbank.com,London,Premium,Savings
C-1006,Liam Murray,liam.new@northbank.com,Leeds,Retail,Savings
C-1009,Priya Singh,priya.singh@northbank.com,Cardiff,Retail,Current


### Task 2.3 — Apply SCD Type 2: Step 1 — Close changed records

Write a `MERGE INTO` statement that compares `staging_customers` against the current versions in `dim_customer`. For any record where `city`, `segment`, `email`, or `account_type` has changed, **close the current version** by setting:
- `valid_to = current_timestamp()`
- `is_current = false`


In [0]:
spark.sql("""
    MERGE INTO dp750_lab_06.silver.dim_customer AS target
    USING staging_customers AS source
    ON target.customer_id = source.customer_id AND target.is_current = true
    WHEN MATCHED AND (
        target.city <> source.city OR
        target.segment <> source.segment OR
        target.email <> source.email OR
        target.account_type <> source.account_type
    )
    THEN UPDATE SET
        target.valid_to = current_timestamp(),
        target.is_current = false
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### Task 2.3 — Apply SCD Type 2: Step 2 — Insert new versions and new customers

Write an `INSERT INTO` statement to:
1. Add **new current rows** for customers whose old versions were just closed (C-1001, C-1003, C-1006)
2. Add the **brand-new customer** C-1009 who has no existing record

New rows should have `valid_from = current_timestamp()`, `valid_to = TIMESTAMP '9999-12-31'`, and `is_current = true`.



In [0]:
spark.sql("""
    INSERT INTO dp750_lab_06.silver.dim_customer (
        customer_id, full_name, email, city, segment, account_type, valid_from, valid_to, is_current
    )
    SELECT
        s.customer_id,
        s.full_name,
        s.email,
        s.city,
        s.segment,
        s.account_type,
        current_timestamp(),
        TIMESTAMP '9999-12-31',
        true
    FROM staging_customers s
    WHERE NOT EXISTS (
        SELECT 1
        FROM dp750_lab_06.silver.dim_customer d
        WHERE d.customer_id = s.customer_id AND d.is_current = true
    )
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

### Task 2.4 — Query the full version history of a customer

Write a query to show **all versions** of customer `C-1001` (Emma Hartley), ordered by `valid_from`. You should see:
- Version 1: London / Retail (expired)
- Version 2: Oxford / Premium (current)

.

In [0]:
df = spark.sql("""
    SELECT *
    FROM dp750_lab_06.silver.dim_customer
    WHERE customer_id = 'C-1001'
    ORDER BY valid_from ASC
""")
display(df)

customer_sk,customer_id,full_name,email,city,segment,account_type,valid_from,valid_to,is_current
1,C-1001,Emma Hartley,emma.hartley@northbank.com,London,Retail,Savings,2020-01-01T00:00:00.000Z,2026-06-25T17:08:16.382Z,false
9,C-1001,Emma Hartley,emma.hartley@northbank.com,Oxford,Premium,Savings,2026-06-25T17:08:44.234Z,9999-12-31T00:00:00.000Z,true


### Task 2.5 — Point-in-time customer lookup

Northbank's regulatory reporting team needs to know the **customer profile as it existed on 2020-06-15** — before any of the changes above were applied.

Write a query that returns one row per customer showing their profile on that date. Use the `valid_from` and `valid_to` columns to select the version that was active on 2020-06-15.



In [0]:
df = spark.sql("""
    SELECT *
    FROM dp750_lab_06.silver.dim_customer
    WHERE valid_from <= DATE '2020-06-15'
      AND valid_to > DATE '2020-06-15'
    ORDER BY customer_id
""")
display(df)

customer_sk,customer_id,full_name,email,city,segment,account_type,valid_from,valid_to,is_current
1,C-1001,Emma Hartley,emma.hartley@northbank.com,London,Retail,Savings,2020-01-01T00:00:00.000Z,2026-06-25T17:08:16.382Z,false
2,C-1002,James Weston,james.weston@northbank.com,Manchester,Retail,Current,2020-01-01T00:00:00.000Z,9999-12-31T00:00:00.000Z,true
3,C-1003,Sophia Chen,sophia.chen@northbank.com,Edinburgh,Premium,Savings,2020-01-01T00:00:00.000Z,2026-06-25T17:08:16.382Z,false
4,C-1004,Oliver Banks,oliver.banks@northbank.com,Birmingham,Retail,Current,2020-01-01T00:00:00.000Z,9999-12-31T00:00:00.000Z,true
5,C-1005,Aisha Patel,aisha.patel@northbank.com,London,Business,Business,2020-01-01T00:00:00.000Z,9999-12-31T00:00:00.000Z,true
6,C-1006,Liam Murray,liam.murray@northbank.com,Leeds,Retail,Savings,2020-01-01T00:00:00.000Z,2026-06-25T17:08:16.382Z,false
7,C-1007,Charlotte Wright,charlotte.wright@northbank.com,Bristol,Premium,Current,2020-01-01T00:00:00.000Z,9999-12-31T00:00:00.000Z,true
8,C-1008,Noah Thompson,noah.thompson@northbank.com,Glasgow,Retail,Savings,2020-01-01T00:00:00.000Z,9999-12-31T00:00:00.000Z,true


## Exercise 3: Change Data Feed & FCA Audit Trail

Basel III and FCA regulations require Northbank to maintain a **queryable audit trail** of all modifications to transaction data. In this exercise, you load payment transactions, simulate data corrections (a common real-world event), and then query the **Change Data Feed** to produce a full audit log.

### Task 3.1 — Load initial transactions

Run the cell below to insert 10 payment transactions into `fact_transactions`. These represent one day's batch of processed payments.

In [0]:
%sql
INSERT INTO dp750_lab_06.silver.fact_transactions VALUES
  ('TXN-001', 'ACC-001', 'C-1001', 'Credit',  1500.00, 'GBP', '2025-11-01', 'Salary payment'),
  ('TXN-002', 'ACC-001', 'C-1001', 'Debit',    120.50, 'GBP', '2025-11-03', 'Amazon purchase'),
  ('TXN-003', 'ACC-002', 'C-1002', 'Debit',    250.00, 'GBP', '2025-11-05', 'Grocery store'),
  ('TXN-004', 'ACC-003', 'C-1003', 'Credit',  3200.00, 'GBP', '2025-11-07', 'Salary payment'),
  ('TXN-005', 'ACC-003', 'C-1003', 'Transfer', 500.00, 'GBP', '2025-11-10', 'Rent transfer'),
  ('TXN-006', 'ACC-004', 'C-1004', 'Debit',     89.99, 'GBP', '2025-11-12', 'Utility bill'),
  ('TXN-007', 'ACC-005', 'C-1005', 'Debit',   1500.00, 'GBP', '2025-11-14', 'Business expense'),
  ('TXN-008', 'ACC-006', 'C-1006', 'Credit',  2800.00, 'GBP', '2025-11-15', 'Salary payment'),
  ('TXN-009', 'ACC-007', 'C-1007', 'Debit',    340.00, 'GBP', '2025-11-18', 'Car insurance'),
  ('TXN-010', 'ACC-008', 'C-1008', 'Transfer', 200.00, 'GBP', '2025-11-20', 'Transfer to savings');

num_affected_rows,num_inserted_rows
10,10


### Task 3.2 — Simulate transaction corrections

Northbank's reconciliation team identified two amount discrepancies. Run the cell below to apply the corrections.

In [0]:
%sql
-- Correction 1: TXN-003 amount was mis-keyed (correct: £275.50, not £250.00)
UPDATE dp750_lab_06.silver.fact_transactions
SET amount = 275.50, description = 'Grocery store (corrected)'
WHERE transaction_id = 'TXN-003';

-- Correction 2: TXN-007 amount is under dispute
UPDATE dp750_lab_06.silver.fact_transactions
SET amount = 1450.00, description = 'Business expense (disputed amount)'
WHERE transaction_id = 'TXN-007';

num_affected_rows
1


### Task 3.3 — Query the Change Data Feed to build an audit trail

Write a query using `table_changes()` to retrieve **all change events** recorded for `fact_transactions` since the beginning of its history. Include the `_change_type`, `_commit_version`, and `_commit_timestamp` metadata columns.



In [0]:
df = spark.sql("""
    SELECT
        transaction_id,
        amount,
        description,
        _change_type,
        _commit_version,
        _commit_timestamp
    FROM table_changes('dp750_lab_06.silver.fact_transactions', 0)
    ORDER BY _commit_timestamp, transaction_id
""")
display(df)

transaction_id,amount,description,_change_type,_commit_version,_commit_timestamp
TXN-001,1500.00,Salary payment,insert,1,2026-06-25T17:09:26.000Z
TXN-002,120.50,Amazon purchase,insert,1,2026-06-25T17:09:26.000Z
TXN-003,250.00,Grocery store,insert,1,2026-06-25T17:09:26.000Z
TXN-004,3200.00,Salary payment,insert,1,2026-06-25T17:09:26.000Z
TXN-005,500.00,Rent transfer,insert,1,2026-06-25T17:09:26.000Z
TXN-006,89.99,Utility bill,insert,1,2026-06-25T17:09:26.000Z
TXN-007,1500.00,Business expense,insert,1,2026-06-25T17:09:26.000Z
TXN-008,2800.00,Salary payment,insert,1,2026-06-25T17:09:26.000Z
TXN-009,340.00,Car insurance,insert,1,2026-06-25T17:09:26.000Z
TXN-010,200.00,Transfer to savings,insert,1,2026-06-25T17:09:26.000Z


### Task 3.4 — Isolate corrected records for FCA reporting

For FCA reporting, the compliance team needs a view showing only the **post-correction state** of modified transactions. Filter the Change Data Feed to show only `update_postimage` records.



In [0]:
df = spark.sql("""
    SELECT
        transaction_id,
        amount,
        description,
        _change_type,
        _commit_version,
        _commit_timestamp
    FROM table_changes('dp750_lab_06.silver.fact_transactions', 0)
    WHERE _change_type = 'update_postimage'
    ORDER BY _commit_timestamp, transaction_id
""")
display(df)

transaction_id,amount,description,_change_type,_commit_version,_commit_timestamp
TXN-003,275.50,Grocery store (corrected),update_postimage,2,2026-06-25T17:09:42.000Z
TXN-007,1450.00,Business expense (disputed amount),update_postimage,3,2026-06-25T17:09:46.000Z


## Exercise 4: Delta Lake Time Travel

Delta Lake maintains a full **transaction log** of every change made to a table. This gives you the ability to query **previous versions** of your data and even **restore** a table to an earlier state — without restoring from backup. This is critical for Northbank when transaction data is accidentally modified.

### Task 4.1 — Inspect the transaction history

Use `DESCRIBE HISTORY` to view the complete operation log of `fact_transactions`. The output shows every version of the table, including the operation type, timestamp, and user.



In [0]:
df = spark.sql("""
    DESCRIBE HISTORY dp750_lab_06.silver.fact_transactions
""")
display(df)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-06-25T17:09:49.000Z,144364009768812,admin@vbhvprksh1outlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4106910360413105),353acd03-13d3-45f1-91e2-9b522876060a,0625-170224-fo5jb4or-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 10170, p25FileSize -> 3810, numDeletionVectorsRemoved -> 1, minFileSize -> 3810, numAddedFiles -> 1, maxFileSize -> 3810, p75FileSize -> 3810, p50FileSize -> 3810, numAddedBytes -> 3810)",null,Databricks-Runtime/18.2.x-photon-scala2.13
3,2026-06-25T17:09:46.000Z,144364009768812,admin@vbhvprksh1outlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(transaction_id#15026 = TXN-007)""])",null,List(4106910360413105),353acd03-13d3-45f1-91e2-9b522876060a,0625-170224-fo5jb4or-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 1, executionTimeMs -> 2903, numDeletionVectorsUpdated -> 1, scanTimeMs -> 983, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 3658, rewriteTimeMs -> 1919)",null,Databricks-Runtime/18.2.x-photon-scala2.13
2,2026-06-25T17:09:42.000Z,144364009768812,admin@vbhvprksh1outlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(transaction_id#14019 = TXN-003)""])",null,List(4106910360413105),4d3c60b9-2274-4a0c-b007-5831bc10e310,0625-170224-fo5jb4or-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 1, executionTimeMs -> 3337, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1260, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 3603, rewriteTimeMs -> 2072)",null,Databricks-Runtime/18.2.x-photon-scala2.13
1,2026-06-25T17:09:26.000Z,144364009768812,admin@vbhvprksh1outlook.onmicrosoft.com,WRITE,"Map(partitionBy -> [], clusterBy -> [""customer_id"",""transaction_date""], statsOnLoad -> true, mode -> Append, clusteringOnWriteStatus -> late-stage clustering triggered)",null,List(4106910360413105),83d98667-9686-47dd-8143-718b6bd141d9,0625-170224-fo5jb4or-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 10, numOutputBytes -> 2909)",null,Databricks-Runtime/18.2.x-photon-scala2.13
0,2026-06-25T17:06:35.000Z,144364009768812,admin@vbhvprksh1outlook.onmicrosoft.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [""customer_id"",""transaction_date""], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableChangeDataFeed"":""true"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-25ec1b7f-4c3e-48bf-96bc-c13bd5fad15a"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-be4213a2-3f87-4360-b8f4-3cf7c7d64b27""}, statsOnLoad -> false)",null,List(4106910360413105),4236fb2b-b44b-435d-b916-b421b66e2d6e,0625-170224-fo5jb4or-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.2.x-photon-scala2.13


### Task 4.2 — Query original transaction amounts before corrections

Using `VERSION AS OF`, retrieve `TXN-003` and `TXN-007` **as they appeared after the initial INSERT but before the corrections** were applied.



In [0]:
df = spark.sql("""
    SELECT transaction_id, amount, description
    FROM dp750_lab_06.silver.fact_transactions VERSION AS OF 1
    WHERE transaction_id IN ('TXN-003', 'TXN-007')
""")
display(df)

transaction_id,amount,description
TXN-003,250.00,Grocery store
TXN-007,1500.00,Business expense


### Task 4.3 — Restore the table to its pre-correction state

The reconciliation team has confirmed the corrections were applied in error and must be rolled back. Use `RESTORE TABLE` to revert `fact_transactions` to **version 1** (after the initial load, before any corrections).

After restoring, verify that TXN-003 and TXN-007 show their original amounts.



In [0]:
spark.sql("""
    RESTORE TABLE dp750_lab_06.silver.fact_transactions TO VERSION AS OF 1
""")

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]

In [0]:
spark.sql("""
    RESTORE TABLE dp750_lab_06.silver.fact_transactions TO VERSION AS OF 1
""")

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]

In [0]:
%sql
SELECT transaction_id, amount, description
FROM dp750_lab_06.silver.fact_transactions
WHERE transaction_id IN ('TXN-003', 'TXN-007')
ORDER BY transaction_id

transaction_id,amount,description
TXN-003,250.00,Grocery store
TXN-007,1500.00,Business expense
